# Prerequisites

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

data_root = Path('../data')

import sys
sys.path.append('../')

import utils_functionality.velocity_calculation as vc
from utils_functionality.model_analysis import extract_agg_features;

# Update main_df

df_main.xlsx before drag force update is used to get updated df_main.xlsx dataset

In [2]:
df_main_dragless = pd.read_excel(
    Path(data_root, 'before_drag_force_update', 'df_main.xlsx')
)
df_main = df_main_dragless.copy(deep=True)
df_main_dragless.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   test                             372 non-null    int64  
 1   net_impact                       372 non-null    int64  
 2   splashing                        372 non-null    int64  
 3   splashing_spectrum               372 non-null    int64  
 4   breaking_up                      372 non-null    int64  
 5   rebound                          372 non-null    int64  
 6   one_drop                         372 non-null    int64  
 7   voltage                          372 non-null    float64
 8   long_impulse_duration            372 non-null    int64  
 9   long_impulse_dur_binary          372 non-null    object 
 10  wettability                      372 non-null    object 
 11  roughness                        372 non-null    float64
 12  liquid_density        

## Droplet density

We will use this density only for droplet mass calculation.
For impact only liquid density is used.

$\rho_\text{d} = \phi \rho_\text{p} + (1 - \phi) \rho_\text{l}$

where $\phi$ is a particle volume fraction

In [3]:
df_main['drop_density'] = (
    df_main['volume_fraction'] * df_main['particle_density']
    + (1 - df_main['volume_fraction']) * df_main['liquid_density']
)
print('Densities stats')
display(
    df_main[['liquid_density', 'drop_density']].describe()
)
print('Relative density difference')
display(
    (np.abs(df_main['liquid_density'] - df_main['drop_density'])\
        /df_main['liquid_density']).describe()
)

Densities stats


,liquid_density,drop_density
count,372.000000,372.000000
mean,1116.612903,1114.533602
std,99.672405,99.390216
min,820.000000,827.200000
25%,1000.000000,1000.000000
50%,1180.000000,1162.000000
75%,1180.000000,1181.600000
max,1180.000000,1261.600000


Relative density difference


count    372.000000
mean       0.009202
std        0.016331
min        0.000000
25%        0.000678
50%        0.001356
75%        0.012281
max        0.069153
dtype: float64

## Update velocities

- `free_fall_velocity` - vertical impact velocity by gravity, but NO drag force
- `drag_velocity` - vertical impact velocity by gravity AND drag force
- `velocity` - normal to substrate component of `drag_velocity`.

Latter feature would be used to get `We` and `Re`, as well as `K`-parameter (`We_Re`)

In [4]:
def get_impact_velocity(row):
    impact_velocity = vc.get_impact_velocity(
        height=row['height'],
        drop_diameter=row['droplet_diameter'],
        drop_density=row['drop_density']
    )
    return impact_velocity

df_main['free_fall_velocity'] = df_main['velocity']

# Vertical velocity with drag force
df_main['drag_velocity'] = df_main.apply(
    get_impact_velocity,
    axis=1
)

print('Velocities stats')
display(
    df_main[['free_fall_velocity', 'drag_velocity']].describe()
)
print('Relative velocities difference')
display(
    (np.abs(df_main['free_fall_velocity'] - df_main['drag_velocity'])\
        /df_main['drag_velocity']).describe()
)

Velocities stats


,free_fall_velocity,drag_velocity
count,372.000000,372.000000
mean,3.657667,3.452099
std,1.059988,0.921146
min,1.980571,1.932493
25%,3.961141,3.669732
50%,3.961141,3.750737
75%,3.961141,3.773442
max,5.941712,5.447128


Relative velocities difference


count    372.000000
mean       0.052865
std        0.028900
min        0.013641
25%        0.045719
50%        0.051849
75%        0.058854
max        0.190410
dtype: float64

Consider inclination

In [5]:
df_main['velocity'] = (
    df_main['drag_velocity'] * np.cos(np.deg2rad(df_main['inclination']))
)

print('Norm Velocities stats')
display(
    df_main[['drag_velocity', 'velocity']].describe()
)
print('Relative norm. velocity difference')
display(
    (np.abs(df_main['drag_velocity'] - df_main['velocity'])\
        /df_main['velocity']).describe()
)

Norm Velocities stats


,drag_velocity,velocity
count,372.000000,372.000000
mean,3.452099,3.296581
std,0.921146,1.010935
min,1.932493,1.369415
25%,3.669732,2.647679
50%,3.750737,3.733421
75%,3.773442,3.766652
max,5.447128,5.447128


Relative norm. velocity difference


count    372.000000
mean       0.071075
std        0.144924
min        0.000000
25%        0.000000
50%        0.000000
75%        0.064178
max        0.414214
dtype: float64

In [6]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 33 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   test                             372 non-null    int64  
 1   net_impact                       372 non-null    int64  
 2   splashing                        372 non-null    int64  
 3   splashing_spectrum               372 non-null    int64  
 4   breaking_up                      372 non-null    int64  
 5   rebound                          372 non-null    int64  
 6   one_drop                         372 non-null    int64  
 7   voltage                          372 non-null    float64
 8   long_impulse_duration            372 non-null    int64  
 9   long_impulse_dur_binary          372 non-null    object 
 10  wettability                      372 non-null    object 
 11  roughness                        372 non-null    float64
 12  liquid_density        

In [7]:
def get_relative_diff(a, b):
    return (a - b)/a

features = ['Re', 'We', 'We_Re']

df_main = extract_agg_features(df_main)

features_description = []

for feature in features:
    features_description.append(
        get_relative_diff(
            df_main_dragless[feature], 
            df_main[feature]
        ).describe()
    )
    
display(pd.concat(features_description, axis=1))

display(df_main[['velocity', 'drag_velocity', 'Re', 'We', 'We_Re']].describe())

,Re,We,We_Re
count,372.000000,372.000000,372.000000
mean,0.100153,0.181287,0.121985
std,0.094934,0.155463,0.112763
min,0.014550,0.028887,0.018154
25%,0.048737,0.095099,0.060546
50%,0.057069,0.110881,0.070820
75%,0.102661,0.194783,0.126636
max,0.352036,0.580143,0.418649


,velocity,drag_velocity,Re,We,We_Re
count,372.000000,372.000000,372.000000,372.000000,372.000000
mean,3.296581,3.452099,2948.016721,639.010971,150.940989
std,1.010935,0.921146,4223.877931,363.895574,73.612378
min,1.369415,1.932493,177.680086,82.778199,33.217556
25%,2.647679,3.669732,519.256142,326.931959,99.049846
50%,3.733421,3.750737,638.383952,673.996591,141.341406
75%,3.766652,3.773442,4521.230870,811.367927,210.962850
max,5.447128,5.447128,17139.003063,1835.681743,411.848360


## Save main dataset

In [8]:
# df_main.to_excel(Path(data_root, 'df_main.xlsx'), index=False)

##